# The dynamics of an FTT technology block in GREU

GREU's energy technology block chooses between technologies for an energy service. DREAM's block is a merit
order: every year the shares follow from that year's prices, so a share jumps to its new level in the year the
prices change. This notebook shows an alternative block of our own, built on FTT (Future Technology
Transformations), where the shares are stocks that move gradually:

$$\begin{aligned}
sqT_{l,t} = sqT_{l,t-1} &+ \sum_{ll} (sqT_{l,t-1} + seed_l)\, sqT_{ll,t-1}\, \frac{F_{l,ll}\, room_l}{LifeSpan_{ll}} \\
 &- \sum_{ll} (sqT_{ll,t-1} + seed_{ll})\, sqT_{l,t-1}\, \frac{F_{ll,l}\, room_{ll}}{LifeSpan_l} \\
 &+ \text{calibrated baseline flows}
\end{aligned}$$

* $F_{l,ll}$ is the preference for technology $l$ over $ll$: a logistic function of the cost difference, 0.5 when
  the costs are equal.
* $room_l = 1 - sqT_{l,t-1}/sqTPotential_l$ stops a technology at its potential.
* $LifeSpan$ sets how fast capacity is replaced.
* $seed_l$ (`SQTSEED`) lets a technology with share 0 enter.

The block is calibrated so that DREAM's baseline holds. Here the block runs alone: energy prices, the price of
capital and the demand for the energy service are exogenous. The data have 22 technologies in two units: `in_ETS`
in industry 23001 and `heating` in industry 10030. The notebook `greu_ftt_co2tax.ipynb` has the whole model.

Details: `greu_technology_shares_note.md` and `tools/ftt_block.py`. The block solves in a fraction of a second, so
every experiment below is run in the notebook.

*Example calculations based on example data. The model is GREU (ModelFlow translation) with an FTT block of our own, not DREAM's GreenREFORM-EU.*

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'tools'))
import greu_viz as gv
import ftt_block as fb
pd.set_option('display.width', 200)

In [2]:
mp, base, s3, dream, meta = fb.block_setup()
FIRST, LAST = meta['first'], meta['last']
YEARS = list(range(FIRST, LAST + 1))
UNITS = {'IN_ETS__23001': 'in_ETS, industry 23001', 'HEATING__10030': 'heating, industry 10030'}
# the exogenous variables of the block that the CO2 tax changes in DREAM's S3 solution
SHOCKED = fb.shock_inputs(mp, dream)


def run(df, model=None):
    return (model or mp)(df, FIRST, LAST, silent=1)


def labels(unit):
    lab = gv.technology_labels(base, unit)
    return {l: f'{l}: coal with CCS' if 'captured co2' in t else t.replace(' incl. biongas', '') for l, t in lab.items()}


def movers(unit, frames, n=6, tol=0.002):
    """the technologies whose share moves most from the baseline in any of the runs"""
    b = gv.shares(base, unit, FIRST, LAST)
    mv = pd.concat([(gv.shares(f, unit, FIRST, LAST).reindex(columns=b.columns) - b).abs().max() for f in frames],
                   axis=1).max(axis=1)
    return list(mv[mv > tol].sort_values(ascending=False).index[:n])


def panels(unit, runs, techs=None, colors=None, dash=None, title='', cols=3):
    """one panel per technology, one line per run. runs = {name: frame}"""
    techs = techs or movers(unit, runs.values())
    lab = labels(unit)
    rows = -(-len(techs) // cols)
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[lab[t] for t in techs], horizontal_spacing=0.06,
                        vertical_spacing=0.16)
    colors = colors or [gv.GRAY] + gv.CAT
    for i, t in enumerate(techs):
        for k, (name, df) in enumerate(runs.items()):
            fig.add_trace(go.Scatter(x=YEARS, y=100 * df.loc[YEARS, f'SQT__{t.upper()}__{unit}'], mode='lines', name=name,
                                     legendgroup=name, showlegend=(i == 0),
                                     line=dict(color=colors[k], width=2, dash=(dash or {}).get(name)),
                                     hovertemplate=name + ' %{x}: %{y:.2f} %<extra></extra>'),
                          row=i // cols + 1, col=i % cols + 1)
    fig.update_layout(template=gv.TEMPLATE, height=120 + 250 * rows, hovermode='x unified',
                      title=title or f'Technology shares, {UNITS[unit]} (% of the energy service)',
                      legend=dict(orientation='h', y=-0.12 if rows > 1 else -0.25))
    fig.update_yaxes(rangemode='tozero')
    return fig

## The baseline

With the calibrated terms the block returns DREAM's baseline shares, and the shares of each unit sum to 1.

In [3]:
res0 = run(base)
endo = sorted(mp.endogene)
print(f'{len(endo)} equations per year, {FIRST}-{LAST}')
print(f'largest difference from the baseline: {(res0.loc[YEARS, endo] - base.loc[YEARS, endo]).abs().max().max():.1e}')
res0.loc[[FIRST, 2030, LAST], [c for c in endo if c.startswith('SQTSUM')]].T

336 equations per year, 2021-2050
largest difference from the baseline: 7.1e-15


year,2021,2030,2050
SQTSUM__HEATING__10030,1.0,1.0,1.0
SQTSUM__IN_ETS__23001,1.0,1.0,1.0


## The CO2 tax: gradual in FTT, at once in DREAM's block

The block gets the energy prices, the capital price and the demand for the energy service from DREAM's solution
of the CO2 tax shock (S3: +750, phased in 2025-2030, technology block on). The first chart shows what the tax does
to the cost of each technology in 2030.

* **in_ETS**: the tax does not change the cost of the fossil technologies. Only coal with CCS (t11) changes: it
  gets 0.07 cheaper per PJ through the price of captured CO2. That is about 10 times the spread of the cost.
* **heating**: the fossil technologies get 0.05 to 0.08 more expensive (3 to 5 spreads); electricity, district
  heat and waste do not change.

In [4]:
ftt = run(s3)
fig = make_subplots(rows=1, cols=2, subplot_titles=list(UNITS.values()), horizontal_spacing=0.25)
for j, unit in enumerate(UNITS, 1):
    lab = labels(unit)
    d = pd.Series({lab[l.lower()]: ftt.at[2030, f'CT__{k}'] - base.at[2030, f'CT__{k}'] for l, k in meta['ftt_groups'][unit]})
    d = d.sort_values()
    fig.add_trace(go.Bar(x=d.values, y=d.index, orientation='h', showlegend=False,
                         marker=dict(color=['#1c5cab' if v < 0 else '#e34948' for v in d.values]),
                         hovertemplate='%{y}: %{x:.4f}<extra></extra>'), row=1, col=j)
fig.update_layout(template=gv.TEMPLATE, height=420, title='Change in the cost per PJ in 2030 from the CO2 tax')
fig.show()
sig = pd.DataFrame({UNITS[u]: base.loc[2030, [f'SIGMAT__{k}' for _, k in meta['ftt_groups'][u]]].describe()[['min', 'mean', 'max']]
                    for u in UNITS})
print('spread of the cost (SIGMAT) in 2030:')
sig.round(4)

spread of the cost (SIGMAT) in 2030:


,"in_ETS, industry 23001","heating, industry 10030"
min,0.0030,0.0062
mean,0.0065,0.0162
max,0.0090,0.0335


In [5]:
RUNS = {'Baseline': base, 'DREAM block': dream, 'FTT block': ftt}
DASH = {'DREAM block': 'dot'}
panels('IN_ETS__23001', RUNS, dash=DASH).show()

In [6]:
panels('HEATING__10030', RUNS, dash=DASH).show()

What the charts show:

* **in_ETS**: DREAM's block moves coal with CCS from 3.1 % to its potential of 12.5 % by 2028 and takes the room
  from gasoline and the three coal technologies. FTT moves it to 4.4 % in 2030 and 7.2 % in 2050, and takes a
  little from every other technology in proportion to its share.
* **heating**: DREAM's block hardly moves, although the fossil technologies get clearly more expensive. They are
  at their potential and far from the margin, so they stay. In FTT the costs are equal in the baseline, so every
  cost change counts: coal goes from 12.5 % to 11.0 % in 2050 and electricity (t10) from 2.6 % to 4.3 %.

In [7]:
fb.shares_table(meta, RUNS, [2030, 2050]).mul(100).round(2)

Baseline        DREAM block        FTT block       
                       2030   2050        2030   2050      2030   2050
HEATING__10030 T1     12.50  12.50       12.50  12.50     12.22  11.44
               T2     12.50  12.50       12.50  12.50     12.23  11.52
               T3     12.50  12.50       12.50  12.50     12.29  11.93
               T4     10.00  10.00       10.00  10.00      9.84   9.54
               T5     12.50  12.50       12.50  12.50     12.51  12.66
               T6      4.98   4.54        5.04   4.58      5.54   6.39
               T7     12.50  12.50       12.50  12.50     12.51  12.67
               T8     12.50  12.50       12.36  12.42     12.19  11.00
               T9      7.67   7.88        7.71   7.90      7.97   8.54
               T10     2.35   2.58        2.39   2.60      2.71   4.31
               T11     0.00   0.00        0.00   0.00      0.00   0.00
IN_ETS__23001  T1     12.49  12.49       12.49  12.49     12.34  12.08
               T2     12.19  12.18        6.99   6.99     12.01  11.48
               T3     12.40  12.39       12.29  12.28     12.23  11.89
               T4      9.94   9.93        9.83   9.82      9.81   9.55
               T5      0.25   0.25        0.25   0.25      0.24   0.20
               T6      0.01   0.01        0.01   0.01      0.01   0.01
               T7     12.50  12.50       12.50  12.50     12.34  12.09
               T8     12.40  12.41       11.86  11.86     12.24  11.91
               T11     3.14   3.14       12.50  12.50      4.42   7.19
               T12    12.38  12.39       11.28  11.28     12.22  11.88
               T13    12.30  12.31       10.00  10.01     12.13  11.73

## Turnover: the life span sets the speed

`LifeSpan` is 5 years for all technologies in the data. The chart shows the same shock with a life span of 5, 10
and 20 years. Only the turnover in the share equation is changed; the levelised capital cost `uTKexp` is exogenous
in the block and stays as it is. The block is calibrated again for each life span.

Coal with CCS in in_ETS reaches 7.2 % in 2050 with a life span of 5 years, 5.9 % with 10 years and 4.7 % with 20
years. DREAM's block is at 12.5 % from 2028 whatever the life span.

In [8]:
life_runs = {'DREAM block': dream}
for life in (5, 10, 20):
    m_, b_, s_, _, _ = fb.block_setup(life=life)
    life_runs[f'FTT, life span {life}'] = run(s_, m_)
LIFE_COL = [gv.GRAY] + gv.ORD4[:0:-1]
panels('IN_ETS__23001', life_runs, techs=['t11', 't13', 't12'], colors=LIFE_COL, dash=DASH,
       title='Life span and the speed of the shift, in_ETS, industry 23001 (% of the energy service)').show()
t11 = pd.DataFrame({n: 100 * df.loc[[2025, 2030, 2035, 2040, 2050], 'SQT__T11__IN_ETS__23001'] for n, df in life_runs.items()})
t11.round(2)

,DREAM block,"FTT, life span 5","FTT, life span 10","FTT, life span 20"
year,,,,
2025,5.28,3.27,3.21,3.18
2030,12.50,4.42,3.78,3.46
2035,12.50,5.50,4.38,3.76
2040,12.50,6.32,4.95,4.07
2050,12.50,7.19,5.91,4.66


## Spread: how sharp the choice is

`SIGMAT = eP * uTKexp * pTK` is the spread of the cost. The preference is a logistic function of the cost
difference measured in spreads, so a smaller spread makes the choice sharper. The chart scales `eP` by 0.5, 1 and 2.

* **in_ETS**: the spread hardly matters (7.2 %, 7.2 % and 7.0 % for coal with CCS in 2050). The cost change is
  10 spreads, so the preference is close to 1 in all three cases and the speed is set by the turnover.
* **heating**: the spread matters. Coal ends at 10.5 %, 11.0 % and 11.7 % in 2050.

In [9]:
ep_runs = {}
for sc in (0.5, 1.0, 2.0):
    m_, b_, s_, _, _ = fb.block_setup(ep_scale=sc)
    ep_runs[f'eP x {sc:g}'] = run(s_, m_)
EP_COL = gv.ORD4[:0:-1]
tab = {}
for unit in UNITS:
    techs = movers(unit, ep_runs.values(), n=3)
    panels(unit, ep_runs, techs=techs, colors=EP_COL,
           title=f'Spread and the size of the shift, {UNITS[unit]} (% of the energy service)').show()
    for t in techs:
        c = f'SQT__{t.upper()}__{unit}'
        tab[(UNITS[unit], labels(unit)[t])] = {'baseline 2050': 100 * base.at[2050, c],
                                               **{n + ', 2050': 100 * df.at[2050, c] for n, df in ep_runs.items()}}
pd.DataFrame(tab).T.round(2)

baseline 2050  eP x 0.5, 2050  eP x 1, 2050  eP x 2, 2050
in_ETS, industry 23001  t11: coal with CCS                   3.14            7.22          7.19          6.99
                        t2: gasoline for transport          12.18           11.47         11.48         11.52
                        t13: coal and coke                  12.31           11.72         11.73         11.76
heating, industry 10030 t6: district heat                    4.54            6.73          6.39          5.75
                        t10: electricity                     2.58            4.69          4.31          3.66
                        t8: coal and coke                   12.50           10.46         11.00         11.66

## The seed: how a technology with share 0 enters

Coal with CCS (t11) has share 0 in heating. In FTT a technology grows in proportion to its own share, so from 0 it
never grows. The seed `SQTSEED` is added to the technology's own share when it receives share.

The experiment makes t11 in heating 0.05 cheaper per PJ than the other technologies from 2025, with three values
of the seed for all technologies.

Without a seed the technology never enters. With the default seed of 0.001 it is at 0.1 % in 2030 and 3.3 % in
2050; with 0.01 it is at 1.3 % in 2030 and 9.9 % in 2050. The seed sets the start of the S-curve, so it is a
parameter that matters and has to be chosen.

In [10]:
K = 'T11__HEATING__10030'
seed_runs = {}
for seed in (0.0, 0.001, 0.01):
    m_, b_, s_, _, _ = fb.block_setup(seed=seed)
    x = b_.copy()
    others = b_.loc[2025:, 'CT__T5__HEATING__10030']                 # the common cost of the other technologies
    x.loc[2025:, f'JCT__{K}'] += others - 0.05 - b_.loc[2025:, f'CT__{K}']
    seed_runs[f'seed {seed:g}'] = run(x, m_)
fig = go.Figure()
for k, (name, df) in enumerate(seed_runs.items()):
    fig.add_trace(go.Scatter(x=YEARS, y=100 * df.loc[YEARS, f'SQT__{K}'], mode='lines', name=name,
                             line=dict(color=gv.ORD4[k + 1], width=2), hovertemplate=name + ' %{x}: %{y:.2f} %<extra></extra>'))
fig.update_layout(template=gv.TEMPLATE, height=400, hovermode='x unified', yaxis_title='% of the energy service',
                  title='Share of coal with CCS (t11) in heating when it becomes the cheapest technology in 2025')
fig.show()
pd.DataFrame({n: 100 * df.loc[[2025, 2030, 2035, 2040, 2050], f'SQT__{K}'] for n, df in seed_runs.items()}).round(3)

,seed 0,seed 0.001,seed 0.01
year,,,
2025,0.0,0.016,0.160
2030,0.0,0.142,1.337
2035,0.0,0.396,3.270
2040,0.0,0.893,5.782
2050,0.0,3.310,9.948


## A temporary tax leaves a lasting change

The tax is removed after 2034: from 2035 the block gets the baseline prices again. In DREAM's block the shares
are a function of the prices of the year, so they would be back at the baseline in 2035. In FTT the shares are
stocks, and the way back is as slow as the way out.

Coal with CCS is at 5.3 % when the tax ends in 2034. It is still at 4.7 % in 2040 and 4.2 % in 2050, against 3.1 %
in the baseline.

In [11]:
tmp = s3.copy()
tmp.loc[2035:, SHOCKED] = base.loc[2035:, SHOCKED]
temp_runs = {'Baseline': base, 'FTT, permanent tax': ftt, 'FTT, tax until 2034': run(tmp)}
fig = panels('IN_ETS__23001', temp_runs, techs=['t11', 't13', 't2'], colors=[gv.GRAY, gv.CAT[0], gv.CAT[1]],
             title='A tax that ends in 2034, in_ETS, industry 23001 (% of the energy service)')
fig.add_vline(x=2034.5, line=dict(color='#b4b2a9', dash='dash', width=1))
fig.show()
pd.DataFrame({n: 100 * df.loc[[2030, 2034, 2035, 2040, 2045, 2050], 'SQT__T11__IN_ETS__23001'] for n, df in temp_runs.items()}).round(2)

,Baseline,"FTT, permanent tax","FTT, tax until 2034"
year,,,
2030,3.14,4.42,4.42
2034,3.14,5.30,5.30
2035,3.14,5.50,5.19
2040,3.14,6.32,4.73
2045,3.14,6.87,4.40
2050,3.14,7.19,4.15


## Checks on the shock run

The flows between two technologies are the same term in both share equations, so the sum of the shares is kept.
The calibrated baseline flows are not limited by the potential, so a share can end a little above it: the
largest case is 0.17 percentage points (waste in heating in 2050, 12.67 % against a potential of 12.5 %).

In [12]:
rows = []
for unit, tl in meta['ftt_groups'].items():
    sq = ftt.loc[YEARS, [f'SQT__{k}' for _, k in tl]]
    pot = s3.loc[YEARS, [f'SQTPOTENTIAL__{k}' for _, k in tl]]
    over = sq.to_numpy() - pot.to_numpy()
    i = np.unravel_index(over.argmax(), over.shape)
    rows.append({'unit': UNITS[unit], 'sum of shares, min': ftt.loc[YEARS, f'SQTSUM__{unit}'].min(),
                 'sum of shares, max': ftt.loc[YEARS, f'SQTSUM__{unit}'].max(), 'smallest share': sq.min().min(),
                 'largest share over potential': over.max(), 'technology': tl[i[1]][0].lower(), 'year': YEARS[i[0]]})
pd.DataFrame(rows).set_index('unit').T

unit,"heating, industry 10030","in_ETS, industry 23001"
"sum of shares, min",1.0,1.0
"sum of shares, max",1.0,1.0
smallest share,-0.0,0.000102
largest share over potential,0.001682,0.0
technology,t7,t7
year,2050,2024
